# Terms

> Terms are thin wrappers around sympy symbols

In [ ]:
#| default_exp term

In [ ]:
#| export
import sympy as sp
import numpy as np
import pandas as pd
from scipy.stats import norm
from sympy.printing.latex import translate
import re

In [ ]:
#| export

class Term(sp.Symbol):
  "A symbolic term in an SFC model."
  default_value = np.nan

  def __new__(cls, name, description=None, initial_value=None, lag=0, unlagged=None, *args, **kwargs):
    "Create a term."
    symbol_name = f"{name}" if lag == 0 else f"{name}_{{-{-lag}}}"
    obj = super().__new__(cls, symbol_name)
    obj.base_name = name
    if unlagged is not None: obj.unlagged = unlagged
    obj.description = description or (unlagged.description if unlagged is not None else "")
    obj.initial_value = unlagged.initial_value if unlagged is not None else initial_value if initial_value is not None else obj.__class__.default_value
    obj.lag = lag
    return obj


  def __call__(self, lag):
    if isinstance(lag, (pd.DataFrame, pd.Series)):  # needed because pandas calls callable keys
        return self
    if not isinstance(lag, (int, np.integer)) or lag > 0:
        raise ValueError(f"lag must be a non-positive int, got {lag!r}")
    base = self.unlagged if self.lag else self
    total = self.lag + int(lag)
    return base if total == 0 else type(self)(self.base_name, lag=total, unlagged=base)


  def _latex(self, printer=None):
    # split the name on "_" (subscript) and "^" (superscript), in any order
    name, *rest = re.split(r"([_^])", self.base_name)
    subs, sups = [], []
    for op, val in zip(rest[::2], rest[1::2]):
        (subs if op == "_" else sups).append(translate(val.strip("{}")))
    if self.lag: subs.append(str(self.lag))   # lag joins the subscript
    out = translate(name)
    if subs: out += "_{" + ",".join(subs) + "}"
    if sups: out += "^{" + ",".join(sups) + "}"
    return out

In [ ]:
#| hide
t = Term("t")
assert isinstance(t,sp.Symbol)

# check translation of greek letters, sub- and super-scripts, including lags
assert sp.latex(Term("pi"),mode="inline") == '$\\pi$'
assert sp.latex(Term("pi_1"),mode="inline") == '$\\pi_{1}$'
assert sp.latex(Term("pi^1"),mode="inline") == '$\\pi^{1}$'
assert sp.latex(Term("pi")(-1),mode="inline") == '$\\pi_{-1}$'
assert sp.latex(Term("pi_1")(-2),mode="inline") == '$\\pi_{1,-2}$'

`Term`s are thin wrappers around Sympy `symbol`s, mainly to implement lags and equations. There are three kinds, `Parameter`s,`Variable`s, and `Random`.

In [ ]:
#| export

class Parameter(Term):
  'An exogenous value, not calculated by the solver.'
    
  pass


Declare a parameter with its LaTeX name, and an optional description and initial value. Be sure to store it in a variable for later use in an equation.

In [ ]:
alpha1 = Parameter("alpha_1","Marginal propensity to consume",0.5)

alpha1

alpha_1

In [ ]:
#| hide
assert alpha1.name == "alpha_1"
assert alpha1.description == "Marginal propensity to consume"
assert alpha1.initial_value == 0.5
assert alpha1.lag == 0

In [ ]:
#| export

class Variable(Term):
  "An endogenous value, solved each timestep."

  def __new__(cls, name, description=None, initial_value=None, lag=0, unlagged=None, *args, growth=False, **kwargs):
    obj = super().__new__(cls, name, description, initial_value, lag, unlagged, *args, **kwargs)
    obj.growth = unlagged.growth if unlagged is not None else bool(growth)
    return obj
    
  def __lshift__(self, other):
    return sp.Eq(self, other)

  def _numpycode(self, printer, **kwargs): # this prevents collision with Variable elsewhere
    return printer._print_Symbol(self)
    

The interface for a Variable is the same as for a `Parameter`, defined with a LaTeX name and optional description and initial value:

In [ ]:
Y = Variable("Y","GDP",100)

Y

Y

In [ ]:
#| hide

assert Y.name == "Y"
assert Y.description == "GDP"
assert Y.initial_value == 100
assert not Y.growth

And you can also optionally set `growth=True`. This will prompt `SFCmodel.check_steady_state()` to compare growth rates instead of levels for this variable, so use it if you're trying to automatically find a steady state:

In [ ]:
V = Variable("V",growth=True)

In [ ]:
#| hide

assert V.growth

In [ ]:
#| export

class Random(Term):
  "A random value, generated each timestep."
    
  default_value = None
  generator = np.random.default_rng()

  def __new__(cls, name, description=None, distribution=norm(0,1), initial_value=None, lag=0, unlagged=None, *args, growth=False, **kwargs):
    obj = super().__new__(cls, name, description, initial_value, lag, unlagged, *args, **kwargs)
    
    if not callable(getattr(distribution, "rvs", None)):
        raise TypeError("distribution must have a .rvs function that generates random numbers, such as a scipy.stats distribution.")
    obj.distribution = distribution
    obj.initial_value = initial_value if initial_value is not None else (cls.default_value if cls.default_value is not None else distribution.rvs(random_state=cls.generator))
    
    return obj

  @classmethod
  def set_seed(cls, seed):
      cls.generator = np.random.default_rng(seed)

In [ ]:
eps = Random("varepsilon","Expectation error")

eps

varepsilon

In [ ]:
#| hide

assert isinstance(eps, Random)
assert isinstance(eps, Term)
assert eps.name == "varepsilon"
assert eps.description == "Expectation error"
assert eps.lag == 0

assert eps.distribution.dist.name == 'norm'
assert eps.distribution.mean() == 0
assert eps.distribution.std() == 1

The default distribution is a standard normal, but you can alternatively supply your own `scipy.stats` random variable (anything that a `.rvs()` call will work on) using the `distribution` argument:

In [ ]:
from scipy import stats
phi = Random("varphi","Animal spirits",stats.bernoulli(0.6))

phi

varphi

In [ ]:
#| hide 
assert phi.distribution.dist.name == 'bernoulli'
assert phi.distribution.mean() == 0.6

You can set optionally set the seed for the random number generator like this. (Do so before creating any random variables, because the initial value is populated on creation by default, unless you specify an `initial_value` on the variable on creation or using `Random.default_value` before creation.)

In [ ]:
seed = 42
Random.set_seed(seed)
# alternatively, set Random.generator to a numpy generator

In [ ]:
#| hide

Random.set_seed(42)
assert eps.distribution.rvs(random_state=Random.generator) == 0.30471707975443135

Random.set_seed(42)
assert Random("r1").initial_value == 0.30471707975443135
assert Random("r2").initial_value == -1.0399841062404955

Random.set_seed(42)
assert Random('r3',initial_value=4).initial_value == 4

Random.default_value = 1
assert Random('r3').initial_value == 1

# restore
Random.default_value = None

## Using terms

Instead of setting default values separately for every term, you can alternatively set them for the whole class, such as:

In [ ]:
Parameter.default_value = 0.3
alpha2 = Parameter("alpha_2")

Variable.default_value = 10
G = Variable("G")

In [ ]:
#| hide
assert alpha2.initial_value == 0.3
assert G.initial_value == 10

Or set them for both `Parameters` and `Variables` by using `Term`:

In [ ]:
# Delete the specific defaults if you assigned them, otherwise skip this part
del Parameter.default_value
del Variable.default_value

Term.default_value = 20
gamma = Parameter("gamma")
C = Variable("C")

In [ ]:
#| hide
assert gamma.initial_value == 20
assert C.initial_value == 20

You can add a lag to a term by putting it in parentheses:

In [ ]:
Y(-1)

Y_{-1}

Then use these in equations like any other symbol.

In [ ]:
#| hide
original = Y
lag = Y(-1)

assert lag.unlagged == original
assert lag.base_name == original.name
assert lag.description == original.description
assert lag.lag == -1

assert original(0) == original
assert original(-2) == lag(-1)
assert lag(-1).unlagged == original

Create equations using `<<` in place of the equals sign - this is just a shorthand for creating `Sympy` equations:

In [ ]:
C = Variable("C","Consumption")
I = Variable("I","Investment")
Y = Variable("Y","GDP")

Y << C + I

Eq(Y, C + I)

In [ ]:
#| hide
assert isinstance(Y << C + I, sp.Eq) 
assert (Y << C + I) == sp.Eq(Y,C+I)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()